# **GPT: Overview and Training Process**
GPT (Generative Pre-trained Transformer) is an autoregressive language model designed to generate coherent text by predicting the next word in a sequence. Unlike BERT, which focuses on understanding text, GPT excels at producing human-like responses. It uses a decoder-only Transformer architecture, leveraging previous words to generate contextually relevant outputs. Pretrained on vast datasets, GPT is fine-tuned for tasks like dialogue generation, summarization, and coding assistance.

GPT’s pretraining follows Causal Language Modeling (CLM), predicting the next word in a left-to-right manner. Temperature scaling controls randomness in output, balancing creativity and coherence. Retrieval-Augmented Generation (RAG) enhances accuracy by integrating external knowledge sources. Fine-tuning optimizes GPT for specific applications, improving response quality and domain relevance.

In [ ]:
 #%pip install transformers # --> installing huggingface's transformers library
 #%pip install torch # --> installing the pytorch library
 #%pip install datasets #--> installing the datasets library
 #%pip install faiss-gpu-cu11==1.10.0 # --> installing the faiss library

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 36.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AdamW, Trainer, TrainingArguments
import faiss
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader

In [ ]:
model_name = "gpt2" # use of GPT2 model from huggingface

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

model.eval() # set to evaluation mode so as to simply output predictions - not gain further insights/ modify weights

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## **1. Autoregression in GPT**

GPT follows an autoregressive approach, generating text one token at a time, conditioning each prediction on previously generated tokens. The code enforces _causal masking_, ensuring the model does not access future tokens during training. This directional constraint aligns with human language processing, reinforcing coherence in extended responses.

Autoregression allows GPT to learn from large-scale text corpora and generate fluent, contextually relevant text by predicting each token sequentially. We will witness how autoregression functions in the context of GPT via the code below!

In [ ]:
### GPT Autoregression

# Define a text prompt to start the generation
prompt_text = "Once upon a time in a land far"
tokenizer.pad_token_id = tokenizer.eos_token_id # set the padding token to be that of ending token
# Encode the prompt text
inputs = tokenizer.encode(prompt_text, return_tensors="pt")


generated_tokens = inputs
max_length = 10  # Max length of the generated text

# Generate one token at a time
for i in range(max_length):
    # Get the model's prediction (logits) for the next token
    with torch.no_grad():
        outputs = model(generated_tokens)

    # Get the logits for the last token
    logits = outputs.logits[:, -1, :]

    # Convert logits to probabilities (softmax) and sample from it
    probabilities = torch.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probabilities, num_samples=1)


    # Decode and print the generated token
    next_token = tokenizer.decode(next_token_id.item())
    print(f"Generated token: {next_token}")

    # Append the predicted token to the sequence - reuse predicted output for new text
    generated_tokens = torch.cat((generated_tokens, next_token_id), dim=-1)
    print(f"Sentence after step {i+1}:",[tokenizer.decode(g) for g in generated_tokens][0])

# Decode and print the final generated text
generated_text = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
print(f"\nGenerated Text: {generated_text}")

Generated token:  from
Sentence after step 1: Once upon a time in a land far from
Generated token:  terrestrial
Sentence after step 2: Once upon a time in a land far from terrestrial
Generated token:  life
Sentence after step 3: Once upon a time in a land far from terrestrial life
Generated token:  there
Sentence after step 4: Once upon a time in a land far from terrestrial life there
Generated token:  appears
Sentence after step 5: Once upon a time in a land far from terrestrial life there appears
Generated token:  to
Sentence after step 6: Once upon a time in a land far from terrestrial life there appears to
Generated token:  be
Sentence after step 7: Once upon a time in a land far from terrestrial life there appears to be
Generated token:  the
Sentence after step 8: Once upon a time in a land far from terrestrial life there appears to be the
Generated token:  substance
Sentence after step 9: Once upon a time in a land far from terrestrial life there appears to be the substance
Gener

## **2. Temperature Scaling in GPT**

Temperature is a _hyperparameter_ that controls the randomness of GPT's output. A lower temperature (e.g., 0.2) makes predictions more deterministic, favoring the most probable words, while a higher temperature (e.g., 1.0 or above) introduces variability, allowing for more diverse and creative responses. Adjusting temperature balances coherence and novelty, enabling GPT to generate structured answers or more imaginative text depending on the task requirement.

In [ ]:
input_text = "Hello there! I am very happy to finally be"

input_ids = tokenizer.encode(input_text, return_tensors='pt')

# Temperature scaling function
def apply_temperature(logits, temperature=1.0):
    # Scale logits by temperature before applying softmax
    return logits / temperature

with torch.no_grad():
    outputs = model(input_ids)
logits = outputs.logits[:, -1, :]

# Apply temperature scaling
temperature = 0.01  # You can experiment with different values here
logits_scaled = apply_temperature(logits, temperature)

# Convert logits to probabilities (softmax)
probabilities = torch.softmax(logits_scaled, dim=-1)

# Get the top-k most probable tokens and their probabilities
top_k = 10 # Number of tokens you want to observe
top_k_probabilities, top_k_indices = torch.topk(probabilities, top_k, dim=-1)

# Decode and print the possible next tokens and their probabilities
for i in range(top_k):
    token_id = top_k_indices[0, i].item()
    token = tokenizer.decode([token_id])
    probability = top_k_probabilities[0, i].item()
    print(f"Token: '{token}', Probability: {probability*100:.4f}")

Token: ' able', Probability: 100.0000
Token: ' back', Probability: 0.0000
Token: ')', Probability: 0.0000
Token: '%', Probability: 0.0000
Token: '*', Probability: 0.0000
Token: '(', Probability: 0.0000
Token: '$', Probability: 0.0000
Token: '"', Probability: 0.0000
Token: '&', Probability: 0.0000
Token: '!', Probability: 0.0000


## **3. Retrieval-Augmented Generation (RAG)**

While GPT relies on pre-trained knowledge, it struggles with real-time updates or niche topics. Retrieval-Augmented Generation (RAG) enhances GPT’s performance by integrating external sources (e.g., databases, APIs, or web searches). RAG retrieves relevant documents or data and feeds them into GPT as additional context before generating responses.

This approach reduces hallucinations, improves factual accuracy, and makes GPT more adaptable for research and business applications. The code thus aims to enhances its responses to given inputs by retrieving "external knowledge", comprising of relevant context dynamically.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Text Documents to supply Additional Information
documents = [
    "GPT-2 is a transformer-based language model that generates text by predicting the next word given a sequence of previous words.",
    "GPT-2 uses self-attention mechanisms to understand context.",
    "GPT-2 uses a decoder-only transformer architecture trained on large datasets.",
    "GPT-2 does not understand meaning but generates text by learning statistical patterns in language.",
    "GPT-2 produces human-like text based on probabilities.",
    "GPT-2 was developed by OpenAI and trained on web text data without fine-tuning on specific tasks.",
    "GPT-2 generates coherent and contextually relevant text but can also produce biased or factually incorrect information.",
    "GPT-2 operates autoregressively, meaning it generates one token at a time based on prior tokens.",
    "GPT-2 was initially withheld by OpenAI due to concerns about misuse before being publicly released.",
    "GPT-2 lacks explicit reasoning capabilities and does not have memory beyond its current context window.",
    "GPT-2's training data consists of publicly available internet text, but it does not have direct access to real-time or private data.",
    "GPT-2's self-attention layers allow it to consider relationships between words regardless of their distance in a sentence.",
    "GPT-2 does not require task-specific fine-tuning to generate high-quality text but can benefit from additional training on specialized datasets.",
    "GPT-2 can be used for a variety of applications including text generation, summarization, and dialogue systems.",
    "GPT-2 struggles with long-range coherence and can sometimes lose track of the topic over extended passages.",
    "GPT-2 uses byte-pair encoding (BPE) for tokenization, which helps it handle rare words and subword units effectively.",
    "GPT-2 does not have explicit world knowledge but encodes information implicitly from its training corpus.",
    "GPT-2 can be fine-tuned on smaller datasets to adapt its outputs for specific domains or writing styles.",
    "GPT-2 assigns probabilities to different possible next words, choosing the most likely or sampling based on a temperature setting.",
    "GPT-2 has multiple versions with different parameter sizes, including 117M, 345M, 762M, and 1.5B parameters."
]


In [ ]:
doc_embeddings = np.array(embedder.encode(documents))
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
def RAG_relevant_info(query, top_k=3):
    query_embedding = np.array(embedder.encode([query])) # embed the query in a similar format as the documents have been
    _, indices = index.search(query_embedding, top_k) # search up the top_k most relevant documents to respond to the query
    retrieved_docs = [documents[i] for i in indices[0]]
    return retrieved_docs # return the docs

In [ ]:
input_prompt = "What is GPT-2 about?"
no_rag_prompt = f"Question: {input_prompt}\nAnswer: "

inputs = tokenizer(no_rag_prompt, return_tensors="pt", max_length=512, truncation=True)
output_ids = model.generate(**inputs, max_length=200, pad_token_id=tokenizer.eos_token_id, do_sample=True,
    eos_token_id=tokenizer.eos_token_id, temperature=0.1)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(output_text)

Question: What is GPT-2 about?
Answer:  GPT-2 is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is a new version of the GPT-1 standard.  It is


In [ ]:
context = " ".join(RAG_relevant_info(input_prompt,6))
rag_prompt = f"Context:{context} | Question:{input_prompt}\nAnswer:"

inputs = tokenizer(rag_prompt, return_tensors="pt", max_length=512, truncation=True)
output_ids = model.generate(**inputs, max_length=200, pad_token_id=tokenizer.eos_token_id, do_sample=True,
    eos_token_id=tokenizer.eos_token_id, temperature=0.1)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(output_text)

Context:GPT-2 is a transformer-based language model that generates text by predicting the next word given a sequence of previous words. GPT-2 operates autoregressively, meaning it generates one token at a time based on prior tokens. GPT-2 does not understand meaning but generates text by learning statistical patterns in language. GPT-2 has multiple versions with different parameter sizes, including 117M, 345M, 762M, and 1.5B parameters. GPT-2 assigns probabilities to different possible next words, choosing the most likely or sampling based on a temperature setting. GPT-2 produces human-like text based on probabilities. | Question:What is GPT-2 about?
Answer: GPT-2 is a language model that generates text by predicting the next word given a sequence of previous words. GPT-2 operates autoregressively, meaning it generates one token at a time based on prior tokens. GPT-2


## **4. Fine-Tuning GPT for Specific Applications**
After pretraining, GPT can be fine-tuned on domain-specific data to improve performance in targeted use cases. Fine-tuning involves training the model on curated datasets with supervised learning to align responses with the desired style, accuracy, and format. Additionally, Reinforcement Learning from Human Feedback (RLHF) refines GPT by incorporating human preferences into the reward model. Fine-tuning ensures better relevance in applications like customer service, medical diagnostics, and creative writing, making GPT more specialized and reliable. This code may talke more time to compelete depending on the computing resource.

In [ ]:
# Load the WikiText-103 dataset
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

# Check the size of the dataset
print(f"Train dataset size: {len(dataset['train'])}")
print(f"Val dataset size: {len(dataset['validation'])}")
print(f"Test dataset size: {len(dataset['test'])}")

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Make sure the tokenizer has an EOS token for padding
tokenizer.pad_token = tokenizer.eos_token

sample_text = dataset["test"][3]["text"]  # first example text from the test set
print("Original Text:", sample_text)

In [ ]:
# Truncate the input to simulate causal masking (we mask everything after a certain token)
def causal_masking(input_text, truncate_length=5):
    tokens = tokenizer.encode(input_text)
    truncated_tokens = tokens[:truncate_length]  # Mask everything after this point
    truncated_text = tokenizer.decode(truncated_tokens, skip_special_tokens=True)
    return truncated_text, truncated_tokens

masked_text, masked_tokens = causal_masking(sample_text, truncate_length=10)

print("\nMasked Text:", masked_text)

# Load the pre-trained GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# Tokenize the masked input
input_ids = tokenizer.encode(masked_text, return_tensors="pt")

# Generate predictions from GPT-2 (let it predict the next tokens)
output = model.generate(input_ids, max_length=len(input_ids[0]) + 20, num_return_sequences=1, temperature=0.7)

# Decode the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
generated_text

In [ ]:
train_data = dataset["train"].select(range(10_000))
val_data = dataset["validation"].select(range(2_000))

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Apply tokenization to the training data
train_dataset = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
val_dataset = val_data.map(tokenize_function, batched=True)

In [ ]:
def modify_data(batch):
    input_ids_batch = batch['input_ids']
    inputs = []
    labels = []

    for input_ids in input_ids_batch:
        # Masking the first 5 tokens
        input_part = input_ids[:5]
        label_part = input_ids[5:]

        # Pad the inputs and labels to the right with pad_token_id
        input_part = input_part + [tokenizer.pad_token_id] * (512 - len(input_part))
        label_part = label_part + [tokenizer.pad_token_id] * (512 - len(label_part))

        # Append the masked input and label
        inputs.append(input_part)
        labels.append(label_part)


    return {'input_ids': inputs, 'labels': labels}

train_data = train_dataset.map(modify_data, batched=True)
val_data = val_dataset.map(modify_data, batched=True)

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",         # Output directory for checkpoints and logs
    num_train_epochs=3,             # Number of epochs
    per_device_train_batch_size=8,  # Batch size per device
    per_device_eval_batch_size=8,   # Batch size for evaluation
    warmup_steps=500,               # Number of warmup steps
    weight_decay=0.01,              # Weight decay
    logging_dir="./logs",           # Directory for logs
    logging_steps=10,               # Log every 10 steps
    evaluation_strategy="epoch",    # Evaluate at the end of each epoch
    save_strategy="epoch",         # Save model at the end of each epoch
    load_best_model_at_end=True    # Load the best model at the end of training
)

# Set up Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

In [ ]:
def collate_fn(batch):
    return {key: torch.tensor([item[key] for item in batch]) for key in batch[0]}

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

optimizer = AdamW(model.parameters(), lr=1e-4)

# Training loop for fine-tuning
epochs = 1
for epoch in range(epochs):
    model.train()
    for step, batch in enumerate(train_dataloader):
        # Forward pass
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % 100 == 0:
            print(f"Epoch {epoch+1}, Step {step}, Loss: {loss.item()}")
